# Exploratory Data Analysis

Each **object** in the bucket gets its own section below (markdown header + outputs). Unsupported extensions are skipped.

**Per file we show:** dtypes; percent missing; numeric `describe()`; top value counts for text-like columns.

**Design choices (and what we could add later):**
- Large **CSV** files are profiled from the **first rows only** (cap) so the notebook stays fast; full-scan stats need chunking or BigQuery.
- **Parquet** is read whole (usually fine); **JSONL** uses the same row cap as CSV; **GeoJSON** is flattened to one row per feature.
- Optional next steps if you care: pick a **target column** for supervised EDA, **time** plots if there is a date column, **join keys** across files, or **duplicate** checks on IDs.

In [ ]:
import io
import json
from pathlib import Path

import pandas as pd
from google.cloud import storage
from IPython.display import Markdown, display

BUCKET = "ps-housing-data"
ROW_CAP = 500_000
OUT = (Path("..") / "data" / "DataAttributes.md").resolve()
client = storage.Client()


def read_df(blob: storage.Blob) -> pd.DataFrame | None:
    name = blob.name
    if not name:
        return None
    suf = Path(name).suffix.lower()
    if suf == ".csv":
        with blob.open("rt", encoding="utf-8", newline="") as f:
            return pd.read_csv(f, nrows=ROW_CAP, low_memory=False)
    if suf == ".parquet":
        return pd.read_parquet(io.BytesIO(blob.download_as_bytes()))
    if suf == ".jsonl" or name.lower().endswith(".jsonl"):
        with blob.open("rt", encoding="utf-8") as f:
            return pd.read_json(f, lines=True, nrows=ROW_CAP)
    if suf == ".geojson":
        raw = json.loads(blob.download_as_bytes())
        feats = raw.get("features", [])
        return pd.json_normalize(feats) if feats else pd.DataFrame()
    return None


def summarize_narrative(name: str, df: pd.DataFrame, chunk: bool) -> str:
    cols = ", ".join(df.columns.astype(str).tolist())
    miss = (df.isna().mean() * 100).sort_values(ascending=False)
    miss_txt = (
        "No missing values in this profile."
        if miss.max() == 0
        else "Notable missingness: "
        + ", ".join(f"`{c}` (~{v:.0f}%)" for c, v in miss.head(5).items() if v > 0)
        + "."
    )
    kinds = df.dtypes.astype(str).value_counts().to_dict()
    kind_txt = ", ".join(f"{k} ({v})" for k, v in kinds.items())
    chunk_txt = (
        " Only the beginning of the file was loaded for profiling." if chunk else ""
    )
    return (
        f"## `{name}`\n\n"
        f"About **{len(df):,}** rows and **{df.shape[1]}** columns in this profile.{chunk_txt} "
        f"Columns: {cols}.\n\n"
        f"Column types: {kind_txt}.\n\n"
        f"{miss_txt}\n"
    )


blobs = [b for b in client.list_blobs(BUCKET) if b.name and not b.name.endswith("/")]
sections: list[str] = []
md_top = (
    f"# Data attributes — housing bucket\n\n"
    f"Bucket: `{BUCKET}`. Pandas loads from GCS; CSV and JSONL profiles use up to "
    f"{ROW_CAP:,} leading rows (not full-file stats for huge tables).\n\n"
    f"## Overview\n\n"
    f"There are **{len(blobs)}** objects. Expect a listings + calendar + reviews + "
    f"neighbourhoods pattern: join `listing_id` in calendar/reviews to listing `id` "
    f"where present; align neighbourhood names to the GeoJSON `properties`.\n\n"
    f"Sparse columns with ~100% missing in a profile are often unused fields in this "
    f"export—confirm against your upstream scrape before treating them as errors.\n\n"
    "---\n"
)

for blob in sorted(blobs, key=lambda b: b.name or ""):
    bn = blob.name
    if not bn:
        continue
    display(Markdown(f"## `{bn}`"))
    df = read_df(blob)
    if df is None:
        display(Markdown("_Skipped — use .csv, .parquet, .jsonl, or .geojson._"))
        sections.append(f"## `{bn}`\n\nSkipped (unsupported type).\n")
        continue
    used_cap = Path(bn).suffix.lower() == ".csv" or bn.lower().endswith(".jsonl")
    chunk = used_cap and len(df) >= ROW_CAP
    display(
        Markdown(
            "**Rows (profile):** " + f"{len(df):,}" + (" — capped" if chunk else "")
        )
    )
    display(df.dtypes.to_frame("dtype"))
    display((df.isna().mean() * 100).round(2).to_frame("missing_%"))
    nums = df.select_dtypes(include="number")
    if len(nums.columns):
        display(nums.describe().T)
    for c in df.select_dtypes(include=("object", "string", "category")).columns:
        display(Markdown(f"**Top — `{c}`**"))
        display(df[c].value_counts(dropna=True).head(10))
    sections.append(summarize_narrative(bn, df, chunk))

md_bot = (
    "\n## Regenerating\n\n"
    "Run this notebook from `src/backend/pipelines` so `../data/DataAttributes.md` "
    "resolves correctly. Re-run after uploading new objects.\n"
)
OUT.parent.mkdir(parents=True, exist_ok=True)
OUT.write_text(md_top + "\n\n---\n\n".join(sections) + md_bot)
display(Markdown(f"Wrote **{OUT}**"))